<a href="https://colab.research.google.com/github/grandmeadow/first-repository/blob/main/GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[QUEST1]

1.   구조차이
  *  Transformer는 Encoder와 Decoder가 모두 있으며 번역 등에 사용되지만, GPT는 Decoder 블록만 쌓아서 만든 모델 인코더-디코더 교차 어텐션 레이어가 없음

2.   어텐션 방식
*   GPT는 미래 토큰을 보지 못하도록 Masked Multi-Head Attention만을 사용

0. 실습환경 구축 및 데이터 다운로드

In [14]:
!pip install tensorflow
!pip install tokenizers
!mkdir -p ~/work/transformer_chatbot/data/ && cd ~/work/transformer_chatbot/data/
!wget https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv

--2026-06-23 07:08:43--  https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv [following]
--2026-06-23 07:08:43--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘ChatbotData.csv.1’

ChatbotData.csv.1   100%[===================>] 868.99K  --.-KB/s    in 0.03s   

2026-06-23 07:08:44 (32.5 MB/s) - ‘ChatbotData.csv.1’ saved [889842/889842]



1. 라이브러리 설치/ 데이터 로드

In [15]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
import re
from tokenizers import SentencePieceBPETokenizer

In [16]:
df = pd.read_csv('ChatbotData.csv')
df.head(5)

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


1.2. 전처리 (pretrain 전용 데이터셋 구축)

In [17]:
# pretrain을 위해 Q & A를 붙여 하나의 데이터로
sentences = [f"{q} {a}" for q, a in zip(df['Q'], df['A'])]

# 임시 텍스트 파일로 저장 (SentencePiece 학습용 입력)
with open("chatbot_corpus.txt", "w", encoding="utf-8") as f:
    for line in sentences:
        f.write(line + "\n")

# 토크나이저 정의
VOCAB_SIZE = 10000
MAX_LEN = 40  # Q + A 합친 최대 길이

tokenizer = SentencePieceBPETokenizer()
tokenizer.train(
    files=["chatbot_corpus.txt"],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"])

# 패딩 토큰과 OOV 토큰 ID 확인 (<pad>=0, <unk>=1 예상)
PAD_ID = tokenizer.token_to_id("<pad>")
UNK_ID = tokenizer.token_to_id("<unk>")

# 문장 인코딩 함수
def encode_sentence(text):
    return tokenizer.encode(text).ids

# 전체 데이터 정수 인코딩
encoded_sequences = [encode_sentence(s) for s in sentences]

# 패딩 처리
padded_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    encoded_sequences, maxlen=MAX_LEN, padding='post', truncating='post', value=PAD_ID)

# GPT pretrain 특성상 입력은 t 시점까지, 정답(y)은 t+1 시점의 토큰이 됨
# X: [token1, token2, token3, ...] -> Y: [token2, token3, token4, ...]
X = padded_sequences[:, :-1]
Y = padded_sequences[:, 1:]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (11823, 39)
Y shape: (11823, 39)


1.3. 입력 블럭 구현

In [18]:
class GPTEmbedding(layers.Layer):
    def __init__(self, vocab_size, d_model, max_len):
        super(GPTEmbedding, self).__init__()
        self.token_emb = layers.Embedding(vocab_size, d_model)
        self.pos_emb = layers.Embedding(max_len, d_model)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

1.4. GPT-1 모델구성 / 학습

In [19]:
class TransformerDecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(TransformerDecoderLayer, self).__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = tf.keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, x, training=None):
        # 인과적 마스크 내장 기능(use_causal_mask) 활용
        attn_output = self.mha(query=x, value=x, key=x, use_causal_mask=True)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2

D_MODEL = 256
NUM_HEADS = 4
DFF = 512
NUM_LAYERS = 3
INPUT_LEN = MAX_LEN - 1

inputs = layers.Input(shape=(INPUT_LEN,), dtype=tf.int32)
x = GPTEmbedding(VOCAB_SIZE, D_MODEL, INPUT_LEN)(inputs)

for _ in range(NUM_LAYERS):
    x = TransformerDecoderLayer(D_MODEL, NUM_HEADS, DFF)(x)

outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)

gpt_model = tf.keras.Model(inputs=inputs, outputs=outputs)
gpt_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 모델 구조
gpt_model.summary()

# 모델 학습 및 결과
gpt_model.fit(X, Y, epochs=10, batch_size=64)

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 39)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_embedding_1 (GPTEmbedding)  │ (None, 39, 256)        │     2,569,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_layer_3     │ (None, 39, 256)        │     1,315,840 │
│ (TransformerDecoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_layer_4     │ (None, 39, 256)        │     1,315,840 │
│ (TransformerDecoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_layer_5     │ (None, 39, 256)        │     1,315,840 │
│ (TransformerDecoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 39, 10000)      │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,087,504 (34.67 MB)

 Trainable params: 9,087,504 (34.67 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 48s 116ms/step - accuracy: 0.7608 - loss: 2.2600
Epoch 2/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 12s 63ms/step - accuracy: 0.7885 - loss: 1.6476
Epoch 3/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 12s 63ms/step - accuracy: 0.8099 - loss: 1.2791
Epoch 4/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 11s 62ms/step - accuracy: 0.8247 - loss: 1.0106
Epoch 5/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 12s 62ms/step - accuracy: 0.8404 - loss: 0.8109
Epoch 6/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 12s 63ms/step - accuracy: 0.8626 - loss: 0.6483
Epoch 7/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 12s 63ms/step - accuracy: 0.8879 - loss: 0.5096
Epoch 8/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 12s 64ms/step - accuracy: 0.9112 - loss: 0.3981
Epoch 9/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 20s 62ms/step - accuracy: 0.9306 - loss: 0.3151
Epoch 10/10
185/185 ━━━━━━━━━━━━━━━━━━━━ 12s 66ms/step - accuracy: 0.9446 - loss: 0.2566


1.5. 입력에 따른 출력 생성

In [30]:
def generate_text_sp(model, tokenizer, seed_text, max_gen_len=10):
    seq = tokenizer.encode(seed_text).ids

    for _ in range(max_gen_len):
        padded = tf.keras.preprocessing.sequence.pad_sequences(
            [seq], maxlen=INPUT_LEN, padding='post', value=PAD_ID)

        predictions = model.predict(padded, verbose=0)
        current_word_index = len(seq) - 1
        if current_word_index >= INPUT_LEN:
            current_word_index = INPUT_LEN - 1

        next_token_id = np.argmax(predictions[0, current_word_index, :])

        # 패딩 또는 종료 토큰 나오면 멈춤
        if next_token_id == PAD_ID:
            break

        seq.append(int(next_token_id))

    # SentencePiece 디코딩(정수 -> 텍스트)
    result_text = tokenizer.decode(seq)
    return result_text

print("== 출력 생성 테스트 ==")
test_queries = ["오늘 무슨 요일이야?", "너는 누구니?", "배고프다"]

for q in test_queries:
    reply = generate_reply(gpt_model, tokenizer, question=q)
    print(f"입력: {q}")
    print(f"출력 결과: {reply}")
    print("=" * 30)

== 출력 생성 테스트 ==
입력: 오늘 무슨 요일이야?
출력 결과: 오늘 무슨 요일이야?음밥통 되겠네요.
입력: 너는 누구니?
출력 결과: 너는 누구니?? 바보는 자기한테 바보라고 하지 않아요.
입력: 배고프다
출력 결과: 배고프다 저도 하고 싶네요.


어제 transformer(encoder-decoder)구조로 학습한 것보다 성능이 더 낫다고 생각되지 않아 질문을 하니 제미나이는 원인을 2개 설명하고 있다.

* 첫번째 원인은 docoder만 사용하는 GPT 경우, 어떤 단어 뒤에 통계적으로 가장 자주 나오는 단어를 이어 붙이는 연습을 했기 때문에 데이터 양이 수십억개가 아니면 질문이 끝나고 새로운 문장을 시작할때 다소 뜬금없는 출력하기 쉽다고 함.  

* 또 다른 원인은 GPT는 매개변수와 데이터가 엄청 많을 때 문맥을 통째로 이해하는 강력한 성능을 발휘하는데, 현재 데이터 셋은 11,800개의 문장이고 모델은 3개 레이어로 단순하여 오히려 수렴이 더디고 엉뚱한 문장을 만들어 낼 가능성이 높다고 함

개선 방안으로써 입력 데이터를 구성할때 단순히 공백으로 잇지말고 1)특수 토큰으로 명확한 경계를 주고, 2)모델학습 epochs을 20회로 늘려 다시 시도해보기로 함

2.2 토크나이저

In [21]:
# 원시 코퍼스 임시 저장
raw_sentences = [f"{q} {a}" for q, a in zip(df['Q'], df['A'])]
with open("chatbot_corpus.txt", "w", encoding="utf-8") as f:
    for line in raw_sentences:
        f.write(line + "\n")

VOCAB_SIZE = 10000
MAX_LEN = 50  # 특수 토큰 추가로 길이를 크게 지정

tokenizer = SentencePieceBPETokenizer()
tokenizer.train(
    files=["chatbot_corpus.txt"],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    # <sep> 토큰 사용하여 질문과 답변 경계 구분
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>", "<sep>"])

# 특수 토큰 ID 매핑
PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")
SEP_ID = tokenizer.token_to_id("<sep>")

# GPT학습을 위한 문장 구조화: <bos> 질문 <sep> 답변 <eos>
structured_sentences = [f"<bos>{q}<sep>{a}<eos>" for q, a in zip(df['Q'], df['A'])]

# 정수 인코딩 진행
encoded_sequences = [tokenizer.encode(s).ids for s in structured_sentences]

# 텐서플로우 패딩
padded_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    encoded_sequences, maxlen=MAX_LEN, padding='post', truncating='post', value=PAD_ID)

# 인과적 언어 모델링을 위한 X, Y 분할
X = padded_sequences[:, :-1]
Y = padded_sequences[:, 1:]


2.3 입력블록

In [22]:
class GPTEmbedding(layers.Layer):
    def __init__(self, vocab_size, d_model, max_len):
        super(GPTEmbedding, self).__init__()
        self.token_emb = layers.Embedding(vocab_size, d_model)
        self.pos_emb = layers.Embedding(max_len, d_model)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        # 토큰 임베딩과 위치 임베딩을 더해 위치 정보를 주입합니다.
        return self.token_emb(x) + self.pos_emb(positions)

2.4 GPT 모델구성/학습

In [29]:
class TransformerDecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(TransformerDecoderLayer, self).__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = tf.keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, x, training=None):
        # use_causal_mask=True 옵션으로 마스킹
        attn_output = self.mha(query=x, value=x, key=x, use_causal_mask=True)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2

# 하이퍼파라미터
D_MODEL = 256
NUM_HEADS = 4
DFF = 512
NUM_LAYERS = 3
INPUT_LEN = MAX_LEN - 1

# 모델 설계
inputs = layers.Input(shape=(INPUT_LEN,), dtype=tf.int32)
x = GPTEmbedding(VOCAB_SIZE, D_MODEL, INPUT_LEN)(inputs)

for _ in range(NUM_LAYERS):
    x = TransformerDecoderLayer(D_MODEL, NUM_HEADS, DFF)(x)

outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)

# 모델 생성 및 컴파일
gpt_model = tf.keras.Model(inputs=inputs, outputs=outputs)
gpt_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

print("== Model Summary ==")
gpt_model.summary()  # 캡처 대상 1

print("== Model Training ==")
gpt_model.fit(X, Y, epochs=20, batch_size=64)  # 캡처 대상 2


== Model Summary ==


Model: "functional_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_21 (InputLayer)     │ (None, 49)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_embedding_6 (GPTEmbedding)  │ (None, 49, 256)        │     2,572,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_layer_16    │ (None, 49, 256)        │     1,315,840 │
│ (TransformerDecoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_layer_17    │ (None, 49, 256)        │     1,315,840 │
│ (TransformerDecoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_layer_18    │ (None, 49, 256)        │     1,315,840 │
│ (TransformerDecoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 49, 10000)      │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,090,064 (34.68 MB)

 Trainable params: 9,090,064 (34.68 MB)

 Non-trainable params: 0 (0.00 B)

== Model Training ==
Epoch 1/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 38s 122ms/step - accuracy: 0.7892 - loss: 1.9540
Epoch 2/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.8179 - loss: 1.3445
Epoch 3/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.8301 - loss: 1.1251
Epoch 4/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 79ms/step - accuracy: 0.8397 - loss: 0.9619
Epoch 5/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.8516 - loss: 0.8212
Epoch 6/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.8682 - loss: 0.6902
Epoch 7/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.8874 - loss: 0.5714
Epoch 8/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.9075 - loss: 0.4712
Epoch 9/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.9246 - loss: 0.3940
Epoch 10/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.9370 - loss: 0.3409
Epoch 11/20
185/185 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.9447 - loss: 0.3079
Epoch 12

2.5 입력에 따른 출력생성

In [28]:
def generate_reply(model, tokenizer, question, max_gen_len=20):

    # 입력 문장 구조화: "<bos> 질문 <sep>"
    input_text = f"<bos>{question}<sep>"
    seq = tokenizer.encode(input_text).ids

    for _ in range(max_gen_len):
        # 모델 입력 길이에 맞춰 패딩 조절
        padded = tf.keras.preprocessing.sequence.pad_sequences(
            [seq], maxlen=INPUT_LEN, padding='post', value=PAD_ID)

        predictions = model.predict(padded, verbose=0)

        # 현재 채워진 시퀀스의 가장 마지막 예측 토큰 위치 지정
        current_word_index = len(seq) - 1
        if current_word_index >= INPUT_LEN:
            current_word_index = INPUT_LEN - 1

        # 가장 높은 확률을 가진 토큰 ID 선택
        next_token_id = np.argmax(predictions[0, current_word_index, :])

        # <eos> 또는 패딩 토큰이 나오면 생성 중지
        if next_token_id == EOS_ID or next_token_id == PAD_ID:
            break

        seq.append(int(next_token_id))

    # 전체 텍스트 디코딩 후 <sep> 답변 부만 잘라내기
    full_text = tokenizer.decode(seq)
    if "<sep>" in full_text:
        reply = full_text.split("<sep>")[-1].strip()
    else:
        reply = full_text

    return reply

print("== 출력 생성 테스트 ==")
test_queries = ["오늘 무슨 요일이야?", "너는 누구니?", "배고프다"]

for q in test_queries:
    reply = generate_reply(gpt_model, tokenizer, question=q)
    print(f"입력: {q}")
    print(f"출력 결과: {reply}")
    print("=" * 30)

== 출력 생성 테스트 ==
입력: 오늘 무슨 요일이야?
출력 결과: 오늘 무슨 요일이야? 초심으로 돌아갈 거 같아요.
입력: 너는 누구니?
출력 결과: 너는 누구니? 아직 안 자요.
입력: 배고프다
출력 결과: 배고프다 저도 밥 먹고 싶어요


In [31]:
def generate_reply_sampled(model, tokenizer, question, max_gen_len=20, temperature=0.7, top_p=0.9):
    input_text = f"<bos>{question}<sep>"
    seq = tokenizer.encode(input_text).ids

    for _ in range(max_gen_len):
        padded = tf.keras.preprocessing.sequence.pad_sequences(
            [seq], maxlen=INPUT_LEN, padding='post', value=PAD_ID
        )

        predictions = model.predict(padded, verbose=0)
        current_word_index = min(len(seq) - 1, INPUT_LEN - 1)

        # 현재 시점의 확률 분포 추출
        logits = predictions[0, current_word_index, :]

        # Temperature 적용 (값이 낮을수록 확실한 단어 선택, 높을수록 창의적)
        logits = np.log(logits + 1e-10) / temperature
        exp_logits = np.exp(logits)
        probs = exp_logits / np.sum(exp_logits)

        # Top-p (Nucleus) Filtering
        sorted_indices = np.argsort(probs)[::-1]
        sorted_probs = probs[sorted_indices]
        cumulative_probs = np.cumsum(sorted_probs)

        # 임계값(top_p)을 넘는 단어들만 남김
        sorted_indices_to_remove = cumulative_probs > top_p
        # 첫 번째 단어는 무조건 포함되도록 시프트
        sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1]
        sorted_indices_to_remove[0] = False

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        probs[indices_to_remove] = 0
        probs = probs / np.sum(probs) # 재정규화

        # 샘플링 진행 (무조건 argmax를 하지 않음)
        next_token_id = np.random.choice(len(probs), p=probs)

        if next_token_id == EOS_ID or next_token_id == PAD_ID:
            break

        seq.append(int(next_token_id))

    full_text = tokenizer.decode(seq)
    return full_text.split("<sep>")[-1].strip() if "<sep>" in full_text else full_text

print("== 출력 생성 테스트 ==")
test_queries = ["오늘 무슨 요일이야?", "너는 누구니?", "배고프다"]

for q in test_queries:
    reply = generate_reply(gpt_model, tokenizer, question=q)
    print(f"입력: {q}")
    print(f"출력 결과: {reply}")
    print("=" * 30)

== 출력 생성 테스트 ==
입력: 오늘 무슨 요일이야?
출력 결과: 오늘 무슨 요일이야?음밥통 되겠네요.
입력: 너는 누구니?
출력 결과: 너는 누구니?? 바보는 자기한테 바보라고 하지 않아요.
입력: 배고프다
출력 결과: 배고프다 저도 하고 싶네요.


- 첫번째 GPT 모델(accuracy=0.94416, loss = 0.2566)에서 문장 생성 결과가 좋지 않아 재시도한 두번째 모델(accuracy=0.9516, loss=0.2614) 역시 문장 생성 결과는 큰 차이가 없어 보인다.  입력문장을 특수 토큰을 이용하고 분리하고, epoch을 20까지 늘려 주어도 매끄러운 답변을 생성하지 못하였다.
- 후속 조치로 파라미터 변경(D_MODEL=512, num_head=8,num_layers=4)을 해보았지만 accuracy=0.73491, loss=0.35184는 더 떨어지고 같은 단어가 반복적으로 나오는 등 더 악화된 성능을 보여주었다.    